# 06 COAD Model Report

This notebook reviews trained COAD tumor vs normal RNA expression models. It does not modify the database or retrain models; it only reads existing results from `data/`, `models/`, and `reports/`.

**Research-only caveat:** tumor data comes from `bio_tcga`, while normal data comes from `tcga_coad`. The two sides may have different data-processing pipelines, so the first-version results are suitable only for research exploration and reporting, not clinical diagnosis.


## 1. Load Report Artifacts

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

summary = json.loads((DATA_DIR / "data_summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(REPORTS_DIR / "metrics_summary.csv")
important_genes = pd.read_csv(REPORTS_DIR / "important_genes.csv")
features = pd.read_parquet(DATA_DIR / "coad_tumor_normal_features.parquet")
labels = pd.read_csv(DATA_DIR / "coad_tumor_normal_labels.csv")
test_samples = pd.read_csv(DATA_DIR / "test_samples.csv")

display(Markdown(
    f"""
### Dataset Summary

- Tumor samples: **{summary['tumor_samples']}**
- Normal samples: **{summary['normal_samples']}**
- Shared genes before low-variance filtering: **{summary['shared_genes']}**
- Model feature matrix shape: **{features.shape[0]} samples x {features.shape[1]} genes**
- Tumor source: `{summary['tumor_source']}`
- Normal source: `{summary['normal_source']}`
    """
))

## 2. Metrics

`accuracy` shows overall correctness. `balanced_accuracy` is more appropriate when normal samples are much fewer than tumor samples. `ROC-AUC` and `PR-AUC` show classification separation ability.


In [ ]:
metric_cols = [
    "model", "accuracy", "balanced_accuracy", "f1", "roc_auc", "pr_auc",
    "normal_precision", "normal_recall", "tumor_precision", "tumor_recall",
]
metrics[metric_cols].style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}",
    "normal_precision": "{:.4f}",
    "normal_recall": "{:.4f}",
    "tumor_precision": "{:.4f}",
    "tumor_recall": "{:.4f}",
})

## 3. Plots

The three plots below are the confusion matrix, ROC curve, and PR curve. In research reports, explain these together with the caveat instead of showing near-perfect metrics by themselves.


In [ ]:
for filename in ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]:
    display(Markdown(f"### {filename}"))
    display(Image(filename=str(REPORTS_DIR / filename)))

## 4. What Is Inside The Saved Models?

Each `.joblib` file stores a sklearn `Pipeline` that includes both the preprocessing step and the model step. New samples must be predicted with the same Pipeline, not only the final classifier.


In [ ]:
model_rows = []
loaded_models = {}
for path in sorted(MODELS_DIR.glob("*.joblib")):
    payload = joblib.load(path)
    pipeline = payload["pipeline"]
    loaded_models[payload["model_name"]] = payload
    model_rows.append({
        "model": payload["model_name"],
        "file": path.name,
        "pipeline_steps": " -> ".join(pipeline.named_steps.keys()),
        "classifier": pipeline.named_steps["model"].__class__.__name__,
        "n_features": len(payload["features"]),
        "random_state": payload["random_state"],
    })

pd.DataFrame(model_rows)

## 5. Example Predictions

`prediction` is the model-predicted label. `score_or_probability` is tumor probability for Logistic Regression and Random Forest, and decision score for Linear SVM.


In [ ]:
classes = np.array(["normal", "tumor"])
test_features = features.loc[test_samples["sample_id"]]
preview = test_samples[["sample_id", "label", "source_schema", "sample_type"]].copy()

for name, payload in loaded_models.items():
    pipeline = payload["pipeline"]
    pred = pipeline.predict(test_features)
    preview[f"{name}_prediction"] = classes[pred]
    if hasattr(pipeline, "predict_proba"):
        preview[f"{name}_score_or_probability"] = pipeline.predict_proba(test_features)[:, 1]
    else:
        preview[f"{name}_score_or_probability"] = pipeline.decision_function(test_features)

preview.head(12)

## 6. Important Genes

These genes are predictive features used by the model to distinguish tumor from normal. They are not automatically proven cancer driver genes. A research report should follow up with literature review.


In [ ]:
important_genes.groupby("model").head(15)[[
    "model", "rank", "gene_symbol", "importance_score", "direction", "short_explanation"
]]

## 7. How To Explain This Model

- Look at `balanced_accuracy` first, because there are only 41 normal samples, far fewer than tumor samples.
- If the Random Forest result is 1.0, do not say the model is perfect. A better interpretation is that current source differences may make the task too easy and require external validation.
- Logistic Regression and Linear SVM coefficients are better for report interpretation: positive coefficients mean higher tumor score, and negative coefficients mean higher normal score.
- `important_genes.csv` can be used as a candidate list for literature review. Do not state it as a causal conclusion.
- Before publication or serious presentation, the next step should be TCGA-GTEx Toil / UCSC Xena validation to check whether the model learned cancer biology rather than source or pipeline batch effect.
